# Filling Missing Rushing Yards: Mean vs Team Mean vs Model
**Juan Zavala** · AI for Analytics · `season_stats.csv`, 272 games, one row per game (the winning team)

Claude worked as a **pair programmer**: each part starts with a plain-English discussion, the plan is reviewed, and only then is code written and run.

| Label | Meaning |
|---|---|
| **[ME]** | Code I wrote myself |
| **[CLAUDE]** | Code Claude wrote after I approved its plan |
| **[REVIEW]** | What I changed or questioned in Claude's plan or code |
| **[CHECK]** | A validation step: does the code do what we think it does? |

The Part 1 discussion (before any code) is in [`part1_exploration.md`](part1_exploration.md).

**The answer key stays out of the analysis.** `actual.csv` is first read in Part 7. Everything before that uses only `season_stats.csv`. The method choices and the model-selection rule were fixed before modeling, and the expected errors are committed to git before Part 7 runs.

---
# Part 1: The nulls, and what the null rows have in common
The assignment's section 2 asks for the data to be read in as `stats`, so the exploration uses that name too.

In [1]:
# [ME] Load the data. The first CSV column is an unnamed row ID (12 to 283), so it becomes the index.
import pandas as pd
import numpy as np

pd.set_option('display.width', 200)
stats = pd.read_csv('data/season_stats.csv', index_col=0)
print(stats.shape)
stats.head()

(272, 7)


,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
12,18,Cowboys,Football Team,38,10,131.0,309
13,18,Rams,49ers,21,20,109.0,163
14,18,Chiefs,Chargers,13,12,123.0,154
15,18,Raiders,Broncos,27,14,129.0,244
16,18,Titans,Jaguars,28,20,175.0,168


In [2]:
# [ME] Where are the nulls?
print(stats.isna().sum().to_string())
print()
print('Null rush_yds:', stats['rush_yds'].isna().sum(), 'of', len(stats),
      f"({stats['rush_yds'].isna().mean():.2%})")
print('rush_yds dtype:', stats['rush_yds'].dtype,
      '| all known values whole numbers:', (stats['rush_yds'].dropna() % 1 == 0).all())

week          0
name          0
opp_name      0
tm_score      0
opp_score     0
rush_yds     16
pass_yds      0

Null rush_yds: 16 of 272 (5.88%)
rush_yds dtype: float64 | all known values whole numbers: True


In [3]:
# [CLAUDE] What do the null rows have in common? Start with the week.
null_rows = stats['rush_yds'].isna()
print('Weeks of the null rows:', stats.loc[null_rows, 'week'].value_counts().to_dict())
print('Games in week 12:', (stats['week'] == 12).sum(),
      '| all of them null:', stats.loc[stats['week'] == 12, 'rush_yds'].isna().all(),
      '| any null outside week 12:', stats.loc[stats['week'] != 12, 'rush_yds'].isna().any())

# [CLAUDE] Something every row shares: is each row the winning team?
print('tm_score > opp_score in every row:', (stats['tm_score'] > stats['opp_score']).all(),
      '| ties:', (stats['tm_score'] == stats['opp_score']).sum())
print('(week, name) is unique:', not stats.duplicated(['week', 'name']).any())

stats.loc[null_rows, ['week', 'name', 'opp_name', 'tm_score', 'opp_score', 'rush_yds', 'pass_yds']]

Weeks of the null rows: {12: 16}
Games in week 12: 16 | all of them null: True | any null outside week 12: False
tm_score > opp_score in every row: True | ties: 0
(week, name) is unique: True


,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
104,12,Bears,Vikings,12,10,NaN,217
105,12,Ravens,Chargers,20,10,NaN,177
106,12,Chiefs,Raiders,31,17,NaN,298
107,12,Eagles,Bills,37,34,NaN,200
108,12,Titans,Panthers,17,10,NaN,185
109,12,Giants,Patriots,10,7,NaN,191
110,12,Jaguars,Texans,24,21,NaN,364
111,12,Broncos,Browns,29,12,NaN,134
112,12,Rams,Cardinals,37,14,NaN,229
113,12,Colts,Buccaneers,27,20,NaN,251


In [4]:
# [CHECK] Is week 12 unusual in any other way? Compare it with the other 17 weeks.
from scipy import stats as sps

explore = stats.assign(margin=stats['tm_score'] - stats['opp_score'])
rows = []
for col in ['tm_score', 'opp_score', 'pass_yds', 'margin']:
    wk12, rest = explore.loc[null_rows, col], explore.loc[~null_rows, col]
    rows.append({'column': col, 'week 12 mean': round(wk12.mean(), 2), 'other weeks mean': round(rest.mean(), 2),
                 'Welch t-test p': round(sps.ttest_ind(wk12, rest, equal_var=False).pvalue, 3)})
print(pd.DataFrame(rows).set_index('column').to_string())

# [CHECK] Does rushing change across the season? (If it did, other weeks would be a biased stand-in.)
known = explore[~null_rows]
weekly = [g['rush_yds'].to_numpy() for _, g in known.groupby('week')]
rho = sps.spearmanr(known['week'], known['rush_yds'])
print(f"\nrush_yds vs week: Spearman r = {rho[0]:+.3f} (p = {rho[1]:.2f}) | "
      f"Kruskal-Wallis across weeks p = {sps.kruskal(*weekly).pvalue:.2f}")

           week 12 mean  other weeks mean  Welch t-test p
column                                                   
tm_score          26.44             27.46           0.686
opp_score         14.88             16.21           0.455
pass_yds         233.94            249.42           0.350
margin            11.56             11.25           0.895

rush_yds vs week: Spearman r = +0.025 (p = 0.69) | Kruskal-Wallis across weeks p = 0.12


**What the null rows have in common:** all 16 are in **week 12**, and every week-12 game is null. One whole week of rushing totals is missing, and no other week has a single gap.

Two more facts apply to the whole file:

* **Every row is the winning team.** `tm_score` beats `opp_score` in all 272 rows. So the 16 missing games are wins, and a team's average from its other wins is a like-for-like stand-in.
* **`rush_yds` is a decimal column only because of the nulls.** Every known value is a whole number, which is why the fills must keep it an integer.

**Is week 12 otherwise unusual?** No. Its scores, passing yards and score margins all match the other weeks (every p-value is above 0.3). Rushing yards don't drift across the season either (Spearman r = +0.025). The gap depends on the week alone, and the week is unrelated to rushing, so filling week 12 from the other 17 weeks shouldn't push the estimates up or down. Part 8 checks this against the answer key.

In [5]:
# [CLAUDE] Which columns carry information about rush_yds? (known rows only)
corr = pd.DataFrame({col: sps.pearsonr(known[col], known['rush_yds']) for col in
                     ['tm_score', 'margin', 'pass_yds', 'opp_score', 'week']}, index=['pearson r', 'p-value']).T
print(corr.round(4).to_string())

team_groups = [g['rush_yds'].to_numpy() for _, g in known.groupby('name')]
opp_groups = [g['rush_yds'].to_numpy() for _, g in known.groupby('opp_name')]
team_share = known.groupby('name')['rush_yds'].transform('mean').var(ddof=0) / known['rush_yds'].var(ddof=0)
print(f"\nTeam (name): ANOVA p = {sps.f_oneway(*team_groups).pvalue:.4f} | "
      f"share of variance explained in-sample = {team_share:.3f}")
print(f"Opponent (opp_name): ANOVA p = {sps.f_oneway(*opp_groups).pvalue:.4f}")

# [CHECK] Leakage scan: no column may be computed from rush_yds
print('\nColumns:', list(stats.columns), '-> none is derived from rushing yards (no total-yards column).')

games_per_team = known['name'].value_counts()
print('\nKnown games per team: min', games_per_team.min(), '| max', games_per_team.max())
print('Known games for the 16 teams missing week 12:',
      games_per_team[stats.loc[null_rows, 'name']].sort_values().to_dict())

           pearson r  p-value
tm_score      0.3119   0.0000
margin        0.2733   0.0000
pass_yds     -0.2093   0.0008
opp_score     0.0199   0.7519
week          0.0012   0.9853

Team (name): ANOVA p = 0.0149 | share of variance explained in-sample = 0.191
Opponent (opp_name): ANOVA p = 0.2549

Columns: ['week', 'name', 'opp_name', 'tm_score', 'opp_score', 'rush_yds', 'pass_yds'] -> none is derived from rushing yards (no total-yards column).

Known games per team: min 2 | max 12
Known games for the 16 teams missing week 12: {'Giants': 5, 'Titans': 5, 'Bears': 6, 'Falcons': 6, 'Broncos': 7, 'Jaguars': 8, 'Colts': 8, 'Packers': 8, 'Steelers': 9, 'Rams': 9, 'Chiefs': 10, 'Eagles': 10, 'Dolphins': 10, '49ers': 11, 'Cowboys': 11, 'Ravens': 12}


**What can predict the missing values?**

* **`tm_score` has the strongest link** (r = +0.31).
* **`pass_yds` has a negative one** (r = −0.21): teams that throw more tend to run less.
* **`opp_score` carries almost nothing on its own** (r = +0.02).
* **Team (`name`) matters** (ANOVA p = 0.015). The 19% "variance explained" is in-sample and optimistic, though, because some teams have very few games.
* **Opponent (`opp_name`) doesn't** (p = 0.25).

All of these relationships are modest, so no method will predict rushing yards closely. The question is which one gets closest.

**For the team mean:** the 16 teams missing week 12 have 5 to 12 known games each, so every one has a team average, and no fallback is needed. The smallest samples (Giants and Titans, 5 games) will be the noisiest.

---
# Part 2: Three copies to fill three different ways

In [6]:
# [ME] Section 2 of the assignment: stats2, stats3 and stats4 are independent copies of stats
stats2 = stats.copy()
stats3 = stats.copy()
stats4 = stats.copy()

# [CHECK] Independent copies: no copy shares memory with stats, so filling one can't change another
for name, df in [('stats2', stats2), ('stats3', stats3), ('stats4', stats4)]:
    print(name, '| same object as stats:', df is stats,
          '| shares memory with stats:', np.shares_memory(df['rush_yds'].to_numpy(), stats['rush_yds'].to_numpy()))

# The 16 rows every method must fill (and the only rows that will be scored in Part 8)
fill_rows = stats.index[stats['rush_yds'].isna()]
print('Rows to fill:', len(fill_rows), '| IDs', fill_rows.min(), 'to', fill_rows.max())

stats2 | same object as stats: False | shares memory with stats: False
stats3 | same object as stats: False | shares memory with stats: False
stats4 | same object as stats: False | shares memory with stats: False
Rows to fill: 16 | IDs 104 to 119


---
# Part 4: The simple solution, the overall mean (`stats2`)

### Prompt 1
> "Give me pandas code to replace the null values in the `rush_yds` column with the average of the non-null values in that column."

### Prompt 2, altered so the column stays an integer
> "Same thing, but the `rush_yds` column must stay an int, because yards are whole numbers."

**Claude's approach:** take the average of the 256 known values, round it to a whole number of yards, fill the 16 nulls with it, then convert the column to `int`.

In [7]:
# [CLAUDE] Prompt 1's answer, previewed on a copy so stats2 isn't changed yet
preview = stats2['rush_yds'].fillna(stats2['rush_yds'].mean())
print('dtype after prompt 1:', preview.dtype, '| value used:', preview.loc[fill_rows].unique())

dtype after prompt 1: float64 | value used: [126.89453125]


**[REVIEW]** Prompt 1 fills every gap with 126.89453125 yards, and the column stays a decimal (`float64`). Real rushing totals are whole numbers, which is why prompt 2 asks for an int.

In [8]:
# [CLAUDE] Prompt 2's answer: round the mean to a whole number of yards, fill, then convert to int
mean_yds = round(stats2['rush_yds'].mean())          # 126.89 -> 127
stats2['rush_yds'] = stats2['rush_yds'].fillna(mean_yds).astype(int)

# [CHECK] No nulls left, the column is an integer, and the 256 known values are untouched
assert stats2['rush_yds'].isna().sum() == 0
assert pd.api.types.is_integer_dtype(stats2['rush_yds'])
assert (stats2['rush_yds'].drop(fill_rows) == stats['rush_yds'].drop(fill_rows)).all()
print('Fill value:', mean_yds, '| dtype:', stats2['rush_yds'].dtype)

# [CHECK] Is 127 a reasonable rushing total? Compare it with the known games.
known_yds = stats['rush_yds'].dropna()
print('Known games: median', known_yds.median(), '| middle half', known_yds.quantile(0.25), 'to', known_yds.quantile(0.75),
      f'| 127 is at the {(known_yds < mean_yds).mean():.0%} percentile')
print('Spread (standard deviation): known games', round(known_yds.std(), 1), '| all 272 after filling', round(stats2['rush_yds'].std(), 1))
stats2.loc[fill_rows, ['week', 'name', 'opp_name', 'tm_score', 'rush_yds']].head(4)

Fill value: 127 | dtype: int64
Known games: median 124.0 | middle half 94.5 to 155.25 | 127 is at the 53% percentile
Spread (standard deviation): known games 49.0 | all 272 after filling 47.5


,week,name,opp_name,tm_score,rush_yds
104,12,Bears,Vikings,12,127
105,12,Ravens,Chargers,20,127
106,12,Chiefs,Raiders,31,127
107,12,Eagles,Bills,37,127


**Is 127 reasonable?** As a single number, yes. It sits near the middle of the known games (median 124, middle half 94.5 to 155.25). But all 16 week-12 games get the same 127.

**Pros**
* The simplest possible fix: two lines of code.
* Part 1 showed the gap is unrelated to rushing, so filling with the mean doesn't pull the season average up or down.
* Nothing to tune, so nothing to overfit.

**Cons**
* Ignores everything about the game: who played, how many points they scored, how much they passed.
* Sixteen identical values shrink the spread. The standard deviation drops from 49.0 (known games) to 47.5 (all 272 rows), so any later look at variability would understate it.
* It can't produce a big rushing day or a bad one. Every guess is "average".